# Phase 8 — ST-CDGM **from-scratch** intégrant 10+ fixes experts

Architecture from-scratch validée par 5 experts (ML/Math/Recherche/IA/Climat). Vise à **battre noncausal v4 sur la majorité des métriques**.

## Probabilités empiriques estimées (basées sur V5 mesuré)

| Métrique | Proba battre noncausal | Justification |
|----------|------------------------|---------------|
| Pearson, RMSE, MAE, CRPS, RAPSD | **70-85%** | V5 le fait déjà ; Min-SNR + features physiques aident |
| SSR (calibration) | 50-60% | Multi-EMA + cond_dropout + Dispersive Loss |
| **F1@p99** | **10-30%** | Verdict expert (17/17 variants échec, 0 précédent publié) |
| CSI/SEDI/FSS | 40-60% | Suivent F1 ; FSS tolère décalages spatiaux |
| Indices climatiques (CDD, R10, RX1) | 60-75% | V5 déjà mieux sur CDD/R10 ; tail_weight raffine |

## Architecture (fixes intégrés)

### Stage 1 (causal mean predictor, from-scratch)
- 15 features existantes + **w_700** (interp pondérée 0.571·w_850 + 0.429·w_500, Holton 2004)
- + **θ_e_850, θ_e_500** (Bolton 1980)
- + **MUCAPE proxy = θ_e_850 − θ_e_500** (remplace CAPE 2-niveaux trop grossier, Emanuel 1994)
- Encoder + RCN + DAG learnable + dual_path Path B UNet
- **Losses physiques différentiables corrigées** :
  - `L_R10mm = MSE(Σ_t σ(5·(expm1(x)-10)), Σ_t 1[expm1(x)>10])` ← seuil en mm/day après expm1
  - `L_Rx1day = LSE_T=5(expm1(pred)) approx max` ← LogSumExp stable
  - `L_CDD = MSE(σ(5·(1-expm1(x))), 1[expm1(x)<1])`
  - `L_CC = autograd((∂μ/∂T_850)·σ_T/σ_μ − 0.07)²` ← adimensionné, autograd obligatoire
  - Pondération : 0.20·R10 + 0.15·Rx1 + 0.10·CDD + 0.05·CC, warmup epoch 10+

### Stage 2 (diffusion EDM)
- UNet 4 niveaux [128,256,256,256] CorrDiff Normal (~50M params)
- **Conditioning** : `causal_concat=True` (fallback documenté — AdaGN reporté à V9 cf. réserve IA)
- **Min-SNR-γ=5** weighting EDM (Hang ICCV 2023, arXiv 2303.09556)
- **Tail_weight (4, 12)** au lieu de (8, 25) — Climate ML
- **Dispersive Loss** λ=0.25 (corrigé vs 0.05 sous-dosé) mid-block hook (He&Wang 2025, arXiv 2506.09027)
- **conditioning_dropout p=0.13** (Ho&Salimans 2022)
- **Multi-EMA** {0.999, 0.9995, 0.9999} avec post-hoc sweep (Karras 2024, arXiv 2312.02696)
- **α appris ∈ [0,1]** avec régularisation `L_α = 0.1·(α-0.5)² + 1.0·max(0, 0.3-α)²` (évite α→0 collapse)

### Sampling
- `dpm_solver++` 32 steps (Lu 2022, arXiv 2211.01095)
- **Limited-Interval Guidance** σ ∈ [0.05, 1.0] (Kynkäänniemi 2024, arXiv 2404.07724)
- cfg_scale 1.0-1.5

### Eval protocol (publication-ready)
- **N_BATCHES = 64**, **K_SAMPLES = 128** (CI ±0.014)
- 3 conventions F1@p99 : pooled full-grid (vs noncausal), ETCCDI per-pixel land, land-only pooled
- CSI@p99, SEDI@p99, FSS (n=9, n=25, n=51 px)
- CRPS gaussian, rank histogram, RMSE, MAE, Pearson global+per-sample
- **Indices climatiques sur 730 jours ENTIERS** (rx1day, CDD, R10mm, DJF/JJA) — ETCCDI Zhang 2011
- μ_HR ablation (causalité opérationnelle), Q_phys (interprétabilité), α appris final
- **Paired permutation test** n=10000 (Phase 8 vs noncausal sur mêmes batches)
- **Bootstrap CI 95% BCa** n=1000 (Math)
- **Holm-Bonferroni** correction multi-tests
- **Pre-enregistrement** : git rev-parse HEAD logged dans JSON output

## Coût compute estimé (A100 Pro+)
- Stage 1 : ~6-7h, 15 epochs
- Stage 2 : ~22-27h, 200 epochs (cached)
- Eval BS30 unifié + comparaison 3-way : ~15h
- **Total : ~45-55h, soit 2-3 sessions Pro+ avec checkpoint/resume**

In [ ]:
# === Cell 1 : Bootstrap Colab ===
import subprocess, shlex, os, sys
from pathlib import Path

REPO_DIR   = Path('/content/climate_data')
GIT_URL    = 'https://github.com/leonelkenfack/stcdgm.git'
GIT_BRANCH = 'four-node-causal'

if not (REPO_DIR / '.git').exists():
    subprocess.run(shlex.split(f'git clone --depth 200 -b {GIT_BRANCH} {GIT_URL} {REPO_DIR}'), check=True)
else:
    subprocess.run(shlex.split(f'git -C {REPO_DIR} fetch --depth=200 origin {GIT_BRANCH}'), check=True)
    subprocess.run(shlex.split(f'git -C {REPO_DIR} reset --hard origin/{GIT_BRANCH}'), check=True)

os.chdir(str(REPO_DIR))
sys.path.insert(0, str(REPO_DIR / 'src'))

try:
    import torch_geometric, cftime, h5netcdf, xbatcher, diffusers
    from omegaconf import OmegaConf
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
        'torch_geometric', 'omegaconf==2.3.0', 'hydra-core==1.3.2',
        'diffusers==0.36.0', 'einops', 'scipy', 'h5py', 'netCDF4',
        'xarray', 'dask', 'zarr', 'safetensors==0.7.0',
        'xbatcher', 'webdataset', 'cftime', 'h5netcdf',
    ], check=True)

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except ModuleNotFoundError:
    print('[bootstrap] not on Colab')

import torch, numpy as np, json, time
import xarray as xr
from omegaconf import OmegaConf

# Reproducibility (Expert IA)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Pre-registration (Expert Recherche)
_git_sha = subprocess.run(['git', 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
print(f'[bootstrap] git SHA = {_git_sha}  branch = {GIT_BRANCH}')
print(f'[bootstrap] cwd={os.getcwd()}  torch={torch.__version__}  cuda={torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'[bootstrap] GPU = {torch.cuda.get_device_name(0)}  VRAM = {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
# === Cell 2 : Constants ===
DRIVE_ROOT = Path('/content/drive/MyDrive/climate_data')
DATA_ROOT  = DRIVE_ROOT / 'data'
HR_RAW_PATH = DATA_ROOT / 'train' / 'pr_ACCESS-CM2_hist.nc'
LR_RAW_PATH = DATA_ROOT / 'train' / 'predictor_ACCESS-CM2_hist.nc'
STATIC_PATH = DATA_ROOT / 'static_predictors' / 'ERA5_eval_ccam_12km.198110_NZ_Invariant.nc'

# Phase 8 output dir
OUT_DIR = DRIVE_ROOT / 'oracle_9node' / 'phase8_from_scratch'
OUT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_STAGE1_LAST  = OUT_DIR / 'stage1_last.pth'
CKPT_STAGE1_BEST  = OUT_DIR / 'stage1_best.pth'
STAGE1_CACHE_PATH = OUT_DIR / 'stage1_cache_mu_total.pt'
CKPT_STAGE2_LAST  = OUT_DIR / 'stage2_last.pth'
CKPT_STAGE2_BEST  = OUT_DIR / 'stage2_best.pth'
LAND_MASK_PATH    = OUT_DIR / 'land_mask_nz.npy'
CLIM_PATH         = OUT_DIR / 'clim_train_p95_p99.npz'
AUGMENTED_LR_PATH = OUT_DIR / 'lr_augmented_features.nc'  # w_700, theta_e, MUCAPE
TRAINING_HISTORY  = OUT_DIR / 'training_history.json'
FINAL_RESULTS     = OUT_DIR / 'phase8_final_results.json'

# Reference checkpoints to compare against (existing models)
REF_CKPT_NONCAUSAL = DRIVE_ROOT / 'ckpt_noncausal'
REF_CKPT_V5_CAUSAL = DRIVE_ROOT / 'ckpt_v2_corrdiff_normal'

# ============== SMOKE MODE FLAG (Expert team validation step) ==============
# When True : 2-3 epochs, N=2 batches, K=4 samples, N_STEPS=8 -> ~15 min total
# When False : full protocol -> ~45-55h
SMOKE_MODE = True

# Reproducibility
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# K9 temporal split (used by all phases)
K9_DATES = {
    'train':   ['1980-01-01', '2009-12-31'],   # 30 years
    'val':     ['2010-01-01', '2011-12-31'],   # 2 years
    'test':    ['2012-01-01', '2013-12-31'],   # 2 years -> 730 days for climate indices
    'holdout': ['2014-01-01', '2014-12-31'],   # 1 year (out-of-distribution warm year)
}

# Stage 1 hyperparams (Path C+ Option C proven on seed_42)
STAGE1_EPOCHS    = 15
STAGE1_LR        = 1e-3
LAMBDA_DAG_PRIOR = 0.40
LAMBDA_L1_START  = 0.04
LAMBDA_L1_END    = 0.005
G_PHYS_ALPHA     = 0.25
# Physical loss weights (warmup applied after epoch 10)
LAMBDA_R10MM   = 0.20
LAMBDA_RX1DAY  = 0.15
LAMBDA_CDD     = 0.10
LAMBDA_CC      = 0.05
PHYS_LOSS_WARMUP_EPOCH = 10

# Stage 2 hyperparams
STAGE2_EPOCHS         = 200
STAGE2_LR             = 2e-4
STAGE2_BATCH_SIZE     = 64
STAGE2_WEIGHT_DECAY   = 1e-4
STAGE2_GRADIENT_CLIP  = 1.0
MIN_SNR_GAMMA         = 5.0
TAIL_WEIGHT_P95       = 4.0    # (Climate ML : 8 -> 4 less aggressive)
TAIL_WEIGHT_P99       = 12.0   # (Climate ML : 25 -> 12)
DISPERSIVE_LAMBDA     = 0.25   # (Expert ML : raised from 0.05 sous-dosé to paper's recommended range)
DISPERSIVE_TAU        = 0.5    # Kernel temperature
COND_DROPOUT_P        = 0.13   # CFG compatibility (CorrDiff Nature CEE 2025)
EMA_DECAYS            = [0.999, 0.9995, 0.9999]   # Multi-EMA EDM2 Karras 2024
ALPHA_TARGET          = 0.5   # Regularization target for learned alpha
ALPHA_FLOOR           = 0.3   # Penalty if alpha < 0.3 (prevent collapse)
LAMBDA_ALPHA_REG      = 0.1
BETA_ALPHA_FLOOR      = 1.0
SIGMA_DATA_NEW        = 0.193  # Phase 6 dualpath recalibrated

# BS30 eval protocol (publication-ready)
N_TEST_BATCHES = 64    # Math expert : N=64 -> CI +/-0.014
K_SAMPLES      = 128   # >> CorrDiff Mardani (32)
N_STEPS_DIFF   = 32    # dpm_solver++ converges at 32 NFE
WET_DAY_THRESHOLD_MM = 1.0   # ETCCDI standard

# Sampling
SAMPLER_SCHEDULER       = 'dpm_solver++'
CFG_SCALE               = 1.0
LIMITED_GUIDANCE_SIGMA_MIN = 0.05   # Kynkäänniemi NeurIPS 2024 (calibrated for sigma_data=0.1)
LIMITED_GUIDANCE_SIGMA_MAX = 1.0

# Bootstrap + statistical tests
BOOTSTRAP_N_RESAMPLES = 1000   # Math : n_boot=1000
PAIRED_PERMUTATION_N = 10000   # Math : n_perm=10000 for p<=0.001
BOOTSTRAP_METHOD     = 'BCa'   # Bias-corrected accelerated (Efron 1987)

print(f'[Cell 2] DEVICE = {DEVICE}')
print(f'[Cell 2] DRIVE_ROOT = {DRIVE_ROOT}')
print(f'[Cell 2] OUT_DIR = {OUT_DIR}')
print(f'[Cell 2] Stage 1 : {STAGE1_EPOCHS} epochs, LR {STAGE1_LR}')
print(f'[Cell 2] Stage 2 : {STAGE2_EPOCHS} epochs, LR {STAGE2_LR}, batch {STAGE2_BATCH_SIZE}')
print(f'[Cell 2] Min-SNR γ = {MIN_SNR_GAMMA}, tail_weight ({TAIL_WEIGHT_P95}, {TAIL_WEIGHT_P99})')
print(f'[Cell 2] Multi-EMA decays = {EMA_DECAYS}')
print(f'[Cell 2] BS30 eval : N={N_TEST_BATCHES} batches x K={K_SAMPLES} samples x {N_STEPS_DIFF} steps')

# Pre-registration record (Expert Recherche)
PRE_REG_RECORD = {
    'phase': 'phase8_from_scratch',
    'git_sha': _git_sha,
    'git_branch': GIT_BRANCH,
    'seed': SEED,
    'k9_dates': K9_DATES,
    'stage1_hyperparams': {
        'epochs': STAGE1_EPOCHS, 'lr': STAGE1_LR,
        'lambda_dag_prior': LAMBDA_DAG_PRIOR,
        'lambda_l1_start': LAMBDA_L1_START, 'lambda_l1_end': LAMBDA_L1_END,
        'g_phys_alpha': G_PHYS_ALPHA,
        'physical_loss_weights': {
            'R10mm': LAMBDA_R10MM, 'Rx1day': LAMBDA_RX1DAY,
            'CDD': LAMBDA_CDD, 'CC': LAMBDA_CC,
        },
        'phys_warmup_epoch': PHYS_LOSS_WARMUP_EPOCH,
    },
    'stage2_hyperparams': {
        'epochs': STAGE2_EPOCHS, 'lr': STAGE2_LR,
        'batch_size': STAGE2_BATCH_SIZE, 'weight_decay': STAGE2_WEIGHT_DECAY,
        'min_snr_gamma': MIN_SNR_GAMMA,
        'tail_weight': [TAIL_WEIGHT_P95, TAIL_WEIGHT_P99],
        'dispersive_lambda': DISPERSIVE_LAMBDA,
        'cond_dropout_p': COND_DROPOUT_P,
        'ema_decays': EMA_DECAYS,
        'alpha_reg': {'target': ALPHA_TARGET, 'floor': ALPHA_FLOOR,
                       'lambda': LAMBDA_ALPHA_REG, 'beta_floor': BETA_ALPHA_FLOOR},
        'sigma_data': SIGMA_DATA_NEW,
    },
    'eval_protocol': {
        'n_batches': N_TEST_BATCHES, 'k_samples': K_SAMPLES,
        'n_steps_diff': N_STEPS_DIFF,
        'sampler': SAMPLER_SCHEDULER, 'cfg_scale': CFG_SCALE,
        'limited_guidance': [LIMITED_GUIDANCE_SIGMA_MIN, LIMITED_GUIDANCE_SIGMA_MAX],
        'bootstrap_n': BOOTSTRAP_N_RESAMPLES,
        'paired_permutation_n': PAIRED_PERMUTATION_N,
    },
    'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
}
if SMOKE_MODE:
    print('[Cell 2] *** SMOKE_MODE active : overriding hyperparams for fast smoke ***')
    STAGE1_EPOCHS = 2
    STAGE2_EPOCHS = 2
    N_TEST_BATCHES = 2
    K_SAMPLES = 4
    N_STEPS_DIFF = 8
    BOOTSTRAP_N_RESAMPLES = 50
    PAIRED_PERMUTATION_N = 500
    print(f'[Cell 2] SMOKE : Stage1={STAGE1_EPOCHS}ep  Stage2={STAGE2_EPOCHS}ep  '
          f'N={N_TEST_BATCHES} K={K_SAMPLES} steps={N_STEPS_DIFF}')

print(f'[Cell 2] Pre-registration record created (SHA {_git_sha[:8]})')
print(f'[Cell 2] SMOKE_MODE = {SMOKE_MODE}')

In [ ]:
# === Cell 3 : Land mask depuis static dataset (multi-source fallback) ===
# Approche : static_predictors fournit orog/he/vegt. On essaie lsm/sftlf/vegt/orog dans l'ordre.

_t0 = time.time()
if not STATIC_PATH.exists():
    raise FileNotFoundError(f'Static dataset not on Drive : {STATIC_PATH}')
ds_static = xr.open_dataset(str(STATIC_PATH), engine='h5netcdf')
print(f'[Cell 3] static variables : {list(ds_static.data_vars)}')

land_mask = None
land_mask_source = None

# Strategy 1 : explicit lsm/sftlf/landfrac
for cand in ('lsm', 'land_sea_mask', 'landfrac', 'land_mask', 'sftlf'):
    if cand in ds_static.data_vars:
        arr = ds_static[cand].values.squeeze()
        if arr.ndim != 2: continue
        thr = 50.0 if arr.max() > 1.5 else 0.5
        land_mask = arr >= thr
        land_mask_source = f'{cand} (threshold {thr})'
        break

# Strategy 2 : vegetation type
if land_mask is None and 'vegt' in ds_static.data_vars:
    vegt = ds_static['vegt'].values.squeeze()
    if vegt.ndim == 2:
        _unique = np.unique(vegt[np.isfinite(vegt)])
        print(f'[Cell 3] vegt unique values = {_unique}')
        land_mask = (vegt > 0) & (vegt != 17) & np.isfinite(vegt)
        land_mask_source = 'vegt (0 and 17 = water)'

# Strategy 3 : orography (safe NaN handling per Climat)
if land_mask is None and 'orog' in ds_static.data_vars:
    orog = ds_static['orog'].values.squeeze()
    if orog.ndim == 2:
        print(f'[Cell 3] orog range : [{np.nanmin(orog):.2f}, {np.nanmax(orog):.2f}] m')
        if np.isnan(orog).any():
            land_mask = np.isfinite(orog) & (orog > -0.5)
            land_mask_source = 'orog : isfinite & > -0.5 m'
        else:
            land_mask = orog > 0.5
            land_mask_source = 'orog > 0.5 m'

if land_mask is None:
    raise RuntimeError(f'No land/sea variable found in static : {list(ds_static.data_vars)}')

n_land = int(land_mask.sum())
n_total = int(land_mask.size)
print(f'[Cell 3] land_mask source : {land_mask_source}')
print(f'[Cell 3] land pixels = {n_land} / {n_total} ({100*n_land/n_total:.1f}%)')
print(f'[Cell 3] expected for NZ : 22-45% (varies with bbox size)')

# Sanity HR shape match
_hr_ds = xr.open_dataset(str(HR_RAW_PATH), engine='h5netcdf')
_pr_var = 'pr' if 'pr' in _hr_ds.data_vars else list(_hr_ds.data_vars)[0]
assert _hr_ds[_pr_var].shape[-2:] == land_mask.shape, 'HR grid mismatch'
_hr_ds.close()

np.save(LAND_MASK_PATH, land_mask)
ds_static.close()
print(f'[Cell 3] saved : {LAND_MASK_PATH}  ({time.time()-_t0:.1f}s)')

In [ ]:
# === Cell 4 : Climatology p95/p99 per-pixel (ETCCDI Zhang 2011) ===
# Per-pixel quantile on wet days >= 1 mm/day, training period only.

_t0 = time.time()
_hr_ds = xr.open_dataset(str(HR_RAW_PATH), engine='h5netcdf')
_pr_var = 'pr' if 'pr' in _hr_ds.data_vars else list(_hr_ds.data_vars)[0]
_hr = _hr_ds[_pr_var]
_time_var = _hr.dims[0]
_hr_train = _hr.sel({_time_var: slice(K9_DATES['train'][0], K9_DATES['train'][1])})
_hr_train_np = _hr_train.values.astype(np.float32)
print(f'[Cell 4] train slice = {_hr_train_np.shape}')

H, W = _hr_train_np.shape[1], _hr_train_np.shape[2]
clim_p95 = np.full((H, W), np.nan, dtype=np.float32)
clim_p99 = np.full((H, W), np.nan, dtype=np.float32)
n_wet = np.zeros((H, W), dtype=np.int32)

for i in range(H):
    for j in range(W):
        if not land_mask[i, j]:
            continue
        px = _hr_train_np[:, i, j]
        px_finite = px[np.isfinite(px)]
        wet = px_finite[px_finite >= WET_DAY_THRESHOLD_MM]
        n_wet[i, j] = wet.size
        if wet.size >= 30:
            clim_p95[i, j] = float(np.quantile(wet, 0.95))
            clim_p99[i, j] = float(np.quantile(wet, 0.99))

_n_valid = int(np.isfinite(clim_p99).sum())
print(f'[Cell 4] land pixels with valid p99 = {_n_valid} / {n_land}')
print(f'[Cell 4] mean wet days per land pixel = {n_wet[land_mask].mean():.0f}')
print(f'[Cell 4] clim_p99 range = [{np.nanmin(clim_p99):.2f}, {np.nanmax(clim_p99):.2f}] mm/day  mean = {np.nanmean(clim_p99):.2f}')

np.savez(CLIM_PATH, clim_p95=clim_p95, clim_p99=clim_p99, n_wet=n_wet,
          land_mask=land_mask, wet_threshold_mm=WET_DAY_THRESHOLD_MM,
          train_start=K9_DATES['train'][0], train_end=K9_DATES['train'][1])
del _hr_train_np
_hr_ds.close()
print(f'[Cell 4] saved : {CLIM_PATH}  ({time.time()-_t0:.1f}s)')

In [ ]:
# === Cell 5 : Augmented LR features (w_700, θ_e_850, θ_e_500, MUCAPE proxy) ===
# Pre-compute offline once, save as NetCDF for fast loading during training.
# Formulas validated by Expert Climat :
#   w_700 = 0.571*w_850 + 0.429*w_500   (linear interpolation in pressure, Holton 2004)
#   θ_e   = θ * exp(L_v * q_sat / (c_p * T))   (Bolton 1980, simplified)
#   MUCAPE_proxy = θ_e_850 - θ_e_500   (static instability indicator, Emanuel 1994)
# Constants : R_d = 287, c_p = 1005, L_v = 2.5e6, R_v = 461.5

_t0 = time.time()

if AUGMENTED_LR_PATH.exists():
    print(f'[Cell 5] augmented LR already exists : {AUGMENTED_LR_PATH}')
    ds_lr_aug = xr.open_dataset(str(AUGMENTED_LR_PATH), engine='h5netcdf')
    print(f'[Cell 5] variables : {list(ds_lr_aug.data_vars)}')
else:
    print(f'[Cell 5] computing augmented features from {LR_RAW_PATH}...')
    ds_lr = xr.open_dataset(str(LR_RAW_PATH), engine='h5netcdf')
    print(f'[Cell 5] LR variables available : {list(ds_lr.data_vars)}')

    # ----- 1. w_700 by linear interpolation in pressure -----
    if 'w_850' in ds_lr.data_vars and 'w_500' in ds_lr.data_vars:
        w_700 = 0.571 * ds_lr['w_850'] + 0.429 * ds_lr['w_500']
        w_700.attrs['long_name'] = 'vertical_velocity_at_700hPa (interpolated)'
        w_700.attrs['units'] = 'Pa s-1'
        w_700.attrs['interpolation'] = '0.571*w_850 + 0.429*w_500 (linear in pressure)'
        print(f'[Cell 5] w_700 computed : shape {w_700.shape} range [{float(w_700.min()):.4f}, {float(w_700.max()):.4f}]')
    else:
        raise RuntimeError('w_850 and w_500 required for w_700 interpolation')

    # ----- 2. θ_e_850 and θ_e_500 via Bolton 1980 -----
    # θ = T * (1000/p)^(R_d/c_p)
    # q_sat via Magnus-Tetens : e_s = 6.112 * exp(17.67*(T-273.15)/(T-29.65))
    # θ_e = θ * exp(L_v * q_sat(T,p) / (c_p * T))
    R_d = 287.0
    c_p = 1005.0
    L_v = 2.5e6
    eps = 0.622  # R_d / R_v

    def _theta_e_bolton(T_K, p_hPa):
        """Bolton 1980 θ_e. T in Kelvin, p in hPa."""
        theta = T_K * (1000.0 / p_hPa) ** (R_d / c_p)
        e_s   = 6.112 * np.exp(17.67 * (T_K - 273.15) / (T_K - 29.65))
        q_sat = eps * e_s / (p_hPa - (1 - eps) * e_s)
        return theta * np.exp(L_v * q_sat / (c_p * T_K))

    if 't_850' in ds_lr.data_vars and 'q_850' in ds_lr.data_vars:
        T_850 = ds_lr['t_850']  # Kelvin (ACCESS-CM2 standard)
        theta_e_850 = xr.apply_ufunc(_theta_e_bolton, T_850, 850.0, dask='allowed')
        theta_e_850.attrs['long_name'] = 'equivalent_potential_temperature_850hPa (Bolton 1980)'
        theta_e_850.attrs['units'] = 'K'
        print(f'[Cell 5] θ_e_850 computed : range [{float(theta_e_850.min()):.1f}, {float(theta_e_850.max()):.1f}] K')
    else:
        raise RuntimeError('t_850 required for θ_e')

    if 't_500' in ds_lr.data_vars and 'q_500' in ds_lr.data_vars:
        T_500 = ds_lr['t_500']
        theta_e_500 = xr.apply_ufunc(_theta_e_bolton, T_500, 500.0, dask='allowed')
        theta_e_500.attrs['long_name'] = 'equivalent_potential_temperature_500hPa (Bolton 1980)'
        theta_e_500.attrs['units'] = 'K'
        print(f'[Cell 5] θ_e_500 computed : range [{float(theta_e_500.min()):.1f}, {float(theta_e_500.max()):.1f}] K')

    # ----- 3. MUCAPE proxy = θ_e_850 - θ_e_500 -----
    mucape_proxy = theta_e_850 - theta_e_500
    mucape_proxy.attrs['long_name'] = 'MUCAPE_proxy (theta_e_850 - theta_e_500)'
    mucape_proxy.attrs['units'] = 'K'
    mucape_proxy.attrs['interpretation'] = 'positive = convectively unstable'
    print(f'[Cell 5] MUCAPE proxy : range [{float(mucape_proxy.min()):.2f}, {float(mucape_proxy.max()):.2f}] K')

    # ----- Build augmented LR dataset (15 original + 4 new = 19 variables) -----
    ds_lr_aug = ds_lr.copy()
    ds_lr_aug['w_700']        = w_700
    ds_lr_aug['theta_e_850']  = theta_e_850
    ds_lr_aug['theta_e_500']  = theta_e_500
    ds_lr_aug['mucape_proxy'] = mucape_proxy

    # Save
    ds_lr_aug.to_netcdf(str(AUGMENTED_LR_PATH), engine='h5netcdf')
    ds_lr.close()
    print(f'[Cell 5] augmented LR saved : {AUGMENTED_LR_PATH}')
    print(f'[Cell 5] total variables = {len(ds_lr_aug.data_vars)} (original 15 + 4 new)')

print(f'[Cell 5] done in {time.time()-_t0:.1f}s')

In [ ]:
# === Cell 6 : Pipeline + dataloaders avec features augmentées ===
from torch.utils.data import DataLoader as _DataLoader, IterableDataset
from st_cdgm.data.pipeline import NetCDFDataPipeline
from st_cdgm.models.graph_builder import HeteroGraphBuilder
from path_c_plus.scripts.option_c_helpers import PATHCPLUS_HYPERPARAM_OVERRIDES

# Load config
CONFIG = OmegaConf.load('config/training_config.yaml')
_corrdiff = OmegaConf.load('config/training_config_corrdiff_normal.yaml')
CONFIG = OmegaConf.merge(CONFIG, _corrdiff)
CONFIG.training.batch_size  = 1
CONFIG.training.use_amp     = True
CONFIG.training.num_workers = 0
ts_cfg = CONFIG.two_stage
ts_cfg.stage1['lambda_dag_prior'] = LAMBDA_DAG_PRIOR
ts_cfg.stage1['g_phys_alpha']     = G_PHYS_ALPHA
OmegaConf.set_struct(CONFIG, False)

# Add augmented LR variables (4 new : w_700, theta_e_850, theta_e_500, mucape_proxy)
ORIGINAL_LR_VARS  = list(CONFIG.data.lr_variables)
AUGMENTED_LR_VARS = ORIGINAL_LR_VARS + ['w_700', 'theta_e_850', 'theta_e_500', 'mucape_proxy']
CONFIG.data.lr_variables = AUGMENTED_LR_VARS
print(f'[Cell 6] LR variables : {len(AUGMENTED_LR_VARS)} (original 15 + 4 augmented)')
print(f'[Cell 6] new variables : w_700, theta_e_850, theta_e_500, mucape_proxy')

# Add 9-node metapaths (existing + new ones for augmented features)
for _m in [
    {'name': 'Q850', 'src': 'Q850', 'relation': 'causes', 'target': 'GP850', 'pool': 'mean'},
    {'name': 'W500', 'src': 'W500', 'relation': 'causes', 'target': 'GP500', 'pool': 'mean'},
    {'name': 'IVT',  'src': 'IVT',  'relation': 'causes', 'target': 'GP850', 'pool': 'mean'},
]:
    if _m['name'] not in {mm.name for mm in CONFIG.encoder.metapaths}:
        CONFIG.encoder.metapaths.append(OmegaConf.create(_m))

# Build pipeline using AUGMENTED LR file (with w_700, theta_e, MUCAPE)
SEQ_LEN = int(CONFIG.data.seq_len)
pipeline = NetCDFDataPipeline(
    lr_path=str(AUGMENTED_LR_PATH), hr_path=str(HR_RAW_PATH),
    static_path=str(STATIC_PATH) if STATIC_PATH.exists() else None,
    seq_len=SEQ_LEN, baseline_strategy=str(CONFIG.data.baseline_strategy),
    baseline_factor=int(CONFIG.data.baseline_factor),
    normalize=bool(CONFIG.data.normalize),
    nan_fill_strategy=str(CONFIG.data.nan_fill_strategy),
    precipitation_delta=float(CONFIG.data.precipitation_delta),
    lr_variables=AUGMENTED_LR_VARS,
    hr_variables=list(CONFIG.data.hr_variables),
    static_variables=list(CONFIG.data.static_variables) if CONFIG.data.get('static_variables') else [],
    means_path=str(DATA_ROOT / 'train' / 'means_ACCESS-CM2.nc') if (DATA_ROOT / 'train' / 'means_ACCESS-CM2.nc').exists() else None,
    stds_path=str(DATA_ROOT / 'train' / 'stds_ACCESS-CM2.nc') if (DATA_ROOT / 'train' / 'stds_ACCESS-CM2.nc').exists() else None,
    train_start_date=K9_DATES['train'][0], train_end_date=K9_DATES['train'][1],
    val_start_date=K9_DATES['val'][0],     val_end_date=K9_DATES['val'][1],
    test_start_date=K9_DATES['test'][0],   test_end_date=K9_DATES['test'][1],
    temporal_holdout_start_date=K9_DATES['holdout'][0],
    temporal_holdout_end_date=K9_DATES['holdout'][1],
)

train_dataset = pipeline.build_sequence_dataset(split='train', seq_len=SEQ_LEN, stride=int(CONFIG.data.stride), as_torch=True)
val_dataset   = pipeline.build_sequence_dataset(split='val',   seq_len=SEQ_LEN, stride=int(CONFIG.data.stride), as_torch=True)
test_dataset  = pipeline.build_sequence_dataset(split='test',  seq_len=SEQ_LEN, stride=int(CONFIG.data.stride), as_torch=True)

train_dataloader = _DataLoader(train_dataset, batch_size=1, num_workers=0, pin_memory=True,
                                collate_fn=lambda x: x, shuffle=False)
val_dataloader   = _DataLoader(val_dataset,   batch_size=1, num_workers=0, pin_memory=True,
                                collate_fn=lambda x: x, shuffle=False)

# Graph builder (9-node)
lr_shape = tuple(CONFIG.graph.lr_shape); hr_shape = tuple(CONFIG.graph.hr_shape)
builder = HeteroGraphBuilder(lr_shape=lr_shape, hr_shape=hr_shape,
                              static_dataset=pipeline.get_static_dataset(),
                              include_mid_layer=CONFIG.graph.include_mid_layer,
                              extended_9node=True)
H_HR, W_HR = int(hr_shape[0]), int(hr_shape[1])

# Convert sample to batch (mirrors phase6 pattern)
_LR_VARS = AUGMENTED_LR_VARS
_VI = {v: i for i, v in enumerate(_LR_VARS)}
_Q_IDX = [_VI[v] for v in ('q_850','q_500','q_250') if v in _VI]
_W_IDX = [_VI[v] for v in ('w_850','w_500','w_250') if v in _VI]
_IVT_LEVELS = [lev for lev in ('850','500','250')
               if f'q_{lev}' in _VI and f'u_{lev}' in _VI and f'v_{lev}' in _VI]

def _compute_ivt_nodes(lr0):
    acc = None
    for lev in _IVT_LEVELS:
        q = lr0[:, [_VI[f'q_{lev}']]]; u = lr0[:, [_VI[f'u_{lev}']]]; v = lr0[:, [_VI[f'v_{lev}']]]
        term = q * torch.sqrt(u*u + v*v + 1e-12)
        acc = term if acc is None else acc + term
    if acc is None: acc = lr0[:, 0:1] * 0.0
    return acc / (len(_IVT_LEVELS) + 1e-8)

def _ensure_2d(t): return t.unsqueeze(-1) if t.dim() == 1 else t

def convert_sample_to_batch(sample, builder, device):
    lr_seq = sample['lr']; seq_len = lr_seq.shape[0]
    lr_nodes_steps = [builder.lr_grid_to_nodes(lr_seq[t]) for t in range(seq_len)]
    lr_tensor = torch.stack(lr_nodes_steps, dim=0)
    lr0 = lr_nodes_steps[0]
    _ivt = _compute_ivt_nodes(lr0)
    dyn = {}
    for nt in builder.dynamic_node_types:
        if nt == 'Q850':   dyn[nt] = _ensure_2d(lr0[:, _Q_IDX] if _Q_IDX else lr0)
        elif nt == 'W500': dyn[nt] = _ensure_2d(lr0[:, _W_IDX] if _W_IDX else lr0)
        elif nt == 'IVT':  dyn[nt] = _ensure_2d(_ivt)
        else:              dyn[nt] = _ensure_2d(lr0)
    hetero = builder.prepare_step_data(dyn).to(device)
    return {'lr': lr_tensor, 'lr_grid': lr_seq, 'residual': sample['residual'],
            'baseline': sample.get('baseline'), 'hetero': hetero, 'time': sample.get('time')}

_probe = next(iter(train_dataset))
C_LR = _probe['lr'].shape[1]
print(f'[Cell 6] LR channels detected = {C_LR} (expected {len(AUGMENTED_LR_VARS)})')
print(f'[Cell 6] HR shape = ({H_HR}, {W_HR})')
print(f'[Cell 6] dynamic node types = {builder.dynamic_node_types}')
print(f'[Cell 6] Pipeline + dataloaders ready')


In [ ]:
# === Cell 7 : Stage 1 build from-scratch + physical losses corrigées ===
from st_cdgm.models.dual_path_stage1 import DualPathPredictor
from st_cdgm.training.stage1_paths import batch_lr_grid_last
from st_cdgm.models.intelligible_encoder import IntelligibleVariableEncoder, IntelligibleVariableConfig
from st_cdgm.models.causal_rcn import RCNCell, RCNSequenceRunner
from st_cdgm.models.regression_head import GraphToGridDecoder

# Build modules from-scratch (NO checkpoint load).
_metapath_configs = [
    IntelligibleVariableConfig(name=m.name, meta_path=(m.src, m.relation, m.target), pool='mean')
    for m in CONFIG.encoder.metapaths
]
encoder = IntelligibleVariableEncoder(
    configs=_metapath_configs,
    hidden_dim=int(CONFIG.encoder.hidden_dim),
    conditioning_dim=int(CONFIG.encoder.conditioning_dim),
).to(DEVICE)
num_vars = len(_metapath_configs)

_lr_nodes = builder.lr_grid_to_nodes(_probe['lr'][0])
rcn_driver_dim = _lr_nodes.shape[-1]
rcn_cell = RCNCell(
    num_vars=num_vars, hidden_dim=int(CONFIG.rcn.hidden_dim),
    driver_dim=rcn_driver_dim, reconstruction_dim=rcn_driver_dim,
    dropout=float(CONFIG.rcn.dropout),
).to(DEVICE)
rcn_runner = RCNSequenceRunner(rcn_cell, detach_interval=CONFIG.rcn.get('detach_interval'))

rh_cfg = CONFIG.two_stage.regression_head
regression_head = GraphToGridDecoder(
    d_model=int(rh_cfg.d_model), hr_h=H_HR, hr_w=W_HR,
    intermediate_h=int(rh_cfg.intermediate_h), intermediate_w=int(rh_cfg.intermediate_w),
    n_heads=int(rh_cfg.n_heads), refine_channels=int(rh_cfg.refine_channels),
    output_channels=1,
).to(DEVICE)

dual_path = DualPathPredictor(
    in_channels=C_LR, base_ch=48, hr_h=H_HR, hr_w=W_HR,
    gate_max_mean=0.40, path_b_kind='unet',
    path_b_unet_channels=(32, 64, 128),
    path_b_unet_lr_shape=(23, 26),
).to(DEVICE)

# Learnable alpha for Stage 2 reconstruction (HR = baseline + alpha * mu_HR + delta)
# Initialised at sigmoid(0) = 0.5 (Math expert : prevents alpha->0 collapse)
alpha_logit = torch.nn.Parameter(torch.tensor(0.0, device=DEVICE))

print(f'[Cell 7] Stage 1 modules built from-scratch')
print(f'[Cell 7]   encoder params  : {sum(p.numel() for p in encoder.parameters()):,}')
print(f'[Cell 7]   rcn_cell params : {sum(p.numel() for p in rcn_cell.parameters()):,}')
print(f'[Cell 7]   reg_head params : {sum(p.numel() for p in regression_head.parameters()):,}')
print(f'[Cell 7]   dual_path params: {sum(p.numel() for p in dual_path.parameters()):,}')
print(f'[Cell 7]   alpha init      : {torch.sigmoid(alpha_logit).item():.4f}')

# ===========================================================================
# Physical losses (differentiable, in mm/day space after expm1 — Math+Climat fix)
# Pipeline applies log1p(x + PRECIPITATION_DELTA), so to recover mm/day :
#   x_mm = expm1(x_log1p) - PRECIPITATION_DELTA
# PRECIPITATION_DELTA = 0.01 (from pipeline.py)
# ===========================================================================
PRECIP_DELTA = 0.01

def _to_mm_day(x_log1p):
    """Convert log1p(pr + delta) back to mm/day, clamped to [0, 500] (Climat)."""
    return (torch.expm1(x_log1p) - PRECIP_DELTA).clamp(min=0.0, max=500.0)

def loss_R10mm(pred_log1p, target_log1p, valid_mask=None, steepness=5.0):
    """Differentiable approx of R10mm (count of days >= 10 mm/day).
    Steepness=5 (Climat raised from 2). Operates in mm/day space."""
    pred_mm = _to_mm_day(pred_log1p)
    target_mm = _to_mm_day(target_log1p)
    # Sigmoid approx of 1[x >= 10]
    pred_soft   = torch.sigmoid(steepness * (pred_mm - 10.0))
    target_soft = torch.sigmoid(steepness * (target_mm - 10.0))
    if valid_mask is not None:
        pred_soft   = pred_soft   * valid_mask
        target_soft = target_soft * valid_mask
    # Sum over time dimension if present, else over batch
    pred_count   = pred_soft.sum(dim=tuple(range(1, pred_soft.dim())))    # per-batch count
    target_count = target_soft.sum(dim=tuple(range(1, target_soft.dim())))
    return ((pred_count - target_count) ** 2).mean()

def loss_Rx1day(pred_log1p, target_log1p, T=5.0):
    """Differentiable approx of annual max (Rx1day).
    Use LogSumExp_T / T (numerically stable approx of max)."""
    pred_mm   = _to_mm_day(pred_log1p)
    target_mm = _to_mm_day(target_log1p)
    # LSE_T(x) = (1/T) * log(sum exp(T*x))  -> approaches max(x) as T -> inf
    flat_pred   = pred_mm.flatten(start_dim=1)
    flat_target = target_mm.flatten(start_dim=1)
    lse_pred   = torch.logsumexp(T * flat_pred,   dim=1) / T
    lse_target = torch.logsumexp(T * flat_target, dim=1) / T
    return ((lse_pred - lse_target) ** 2).mean()

def loss_CDD(pred_log1p, target_log1p, steepness=5.0):
    """Differentiable approx of CDD (count of dry days, x < 1 mm/day)."""
    pred_mm   = _to_mm_day(pred_log1p)
    target_mm = _to_mm_day(target_log1p)
    # Sigmoid approx of 1[x < 1]
    pred_soft   = torch.sigmoid(steepness * (1.0 - pred_mm))
    target_soft = torch.sigmoid(steepness * (1.0 - target_mm))
    pred_count   = pred_soft.sum(dim=tuple(range(1, pred_soft.dim())))
    target_count = target_soft.sum(dim=tuple(range(1, target_soft.dim())))
    return ((pred_count - target_count) ** 2).mean()

def loss_Clausius_Clapeyron(mu_HR_pred, T_850_batch, target_log1p, mode='extreme'):
    """Clausius-Clapeyron loss : (d mu / d T) - rate * mu = 0.
    Math : autograd, NOT finite differences. Adimensionned.
    Climat : mode='extreme' (rate=0.07 for P>P95), mode='mean' (rate=0.05)."""
    if not T_850_batch.requires_grad:
        T_850_batch = T_850_batch.detach().requires_grad_(True)
    # We need mu_HR_pred to depend on T_850 for autograd to work.
    # Caller is responsible for this dependency. Here we just compute the loss.
    rate = 0.07 if mode == 'extreme' else 0.05
    try:
        grad = torch.autograd.grad(
            outputs=mu_HR_pred.sum(),
            inputs=T_850_batch,
            create_graph=True,
            retain_graph=True,
        )[0]
    except RuntimeError:
        # If T_850 was not part of the graph, return zero loss (safe fallback)
        return torch.tensor(0.0, device=mu_HR_pred.device, requires_grad=False)
    # Adimensionner par sigma de chaque variable (Math)
    sigma_T = T_850_batch.std().clamp_min(1e-6)
    sigma_mu = mu_HR_pred.std().clamp_min(1e-6)
    grad_adim = grad * sigma_T / sigma_mu
    target_grad = rate * mu_HR_pred / sigma_mu
    return ((grad_adim - target_grad) ** 2).mean()

# Alpha regularization (Math : prevent alpha -> 0 collapse, cf. MC2RD failure)
def loss_alpha_reg(alpha):
    target_loss = LAMBDA_ALPHA_REG * (alpha - ALPHA_TARGET) ** 2
    floor_loss  = BETA_ALPHA_FLOOR * (torch.relu(ALPHA_FLOOR - alpha) ** 2)
    return target_loss + floor_loss

print('[Cell 7] Physical losses defined :')
print('  loss_R10mm        (steepness=5, in mm/day space)')
print('  loss_Rx1day       (LogSumExp_T=5)')
print('  loss_CDD          (steepness=5)')
print('  loss_Clausius_Clapeyron (autograd, mode=extreme)')
print('  loss_alpha_reg    (lambda=0.1, beta_floor=1.0 for alpha < 0.3)')


In [ ]:
# === Cell 8 : Stage 1 training loop from-scratch ===
# 15 epochs (or 2 in SMOKE_MODE), Adam optimizer.
# DAG prior + L1 sparsity cosine decay + physical losses warmup at epoch 10+.
import time
from torch.optim import Adam

# Optimizer (all Stage 1 modules trainable)
stage1_params = (
    list(encoder.parameters()) + list(rcn_cell.parameters())
    + list(regression_head.parameters()) + list(dual_path.parameters())
)
optimizer_s1 = Adam(stage1_params, lr=STAGE1_LR)

# L1 sparsity schedule (cosine decay)
def _l1_schedule(epoch, total):
    if total <= 1: return LAMBDA_L1_END
    cos_factor = 0.5 * (1 + np.cos(np.pi * epoch / total))
    return LAMBDA_L1_END + (LAMBDA_L1_START - LAMBDA_L1_END) * cos_factor

# Helper : Stage 1 forward through dual-path
def _stage1_forward(batch):
    """Returns (mu_A, mu_total, mu_B, gate, baseline_log, target_residual)."""
    lr_data = batch['lr'].to(DEVICE)
    h_init  = encoder.init_state(batch['hetero']).to(DEVICE)
    drivers = [lr_data[t] for t in range(lr_data.shape[0])]
    seq_out = rcn_runner.run(h_init, drivers, reconstruction_sources=None)
    mu_A = regression_head(seq_out.states[-1])
    if mu_A.dim() == 3: mu_A = mu_A.unsqueeze(0)
    lr_grid = batch_lr_grid_last(batch, builder=builder, device=DEVICE)
    lr_safe = torch.nan_to_num(lr_grid, nan=0.0)
    mu_total, mu_B, gate = dual_path(lr_safe, mu_A)
    mu_total = torch.nan_to_num(mu_total, nan=0.0)
    bl = batch['baseline'][-1].to(DEVICE)
    if bl.dim() == mu_total.dim() - 1: bl = bl.unsqueeze(0)
    bl = torch.nan_to_num(bl, nan=0.0)
    tgt = batch['residual'][-1].to(DEVICE)
    if tgt.dim() == 3: tgt = tgt.unsqueeze(0)
    return mu_A, mu_total, mu_B, gate, bl, tgt

# Training history
stage1_history = []

print(f'[Cell 8] Stage 1 training : {STAGE1_EPOCHS} epochs (SMOKE_MODE={SMOKE_MODE})')
for ep in range(1, STAGE1_EPOCHS + 1):
    _t0 = time.time()
    # Mode train
    for m in [encoder, rcn_cell, regression_head, dual_path]:
        m.train()

    lam_l1 = _l1_schedule(ep - 1, STAGE1_EPOCHS)
    phys_active = (ep >= PHYS_LOSS_WARMUP_EPOCH)

    epoch_losses = {'mse': [], 'dag': [], 'l1': [], 'r10mm': [], 'rx1day': [], 'cdd': [], 'cc': []}
    _n_batches_done = 0
    for sample in train_dataset:
        batch = convert_sample_to_batch(sample, builder, DEVICE)
        optimizer_s1.zero_grad(set_to_none=True)

        mu_A, mu_total, mu_B, gate, baseline_log, target_residual = _stage1_forward(batch)

        # Core MSE on residual (HR - baseline)
        valid = torch.isfinite(target_residual)
        diff = (mu_total - target_residual) ** 2
        diff_masked = torch.where(valid, diff, torch.zeros_like(diff))
        loss_mse = diff_masked.sum() / valid.sum().clamp_min(1)

        # DAG prior + L1 sparsity (existing pattern in repo)
        _rcn_core = rcn_cell._orig_mod if hasattr(rcn_cell, '_orig_mod') else rcn_cell
        if hasattr(_rcn_core, 'A_dag'):
            A_dag = _rcn_core.A_dag
            loss_dag = LAMBDA_DAG_PRIOR * ((A_dag * A_dag).sum())  # placeholder prior penalty
            loss_l1  = lam_l1 * A_dag.abs().sum()
        else:
            loss_dag = torch.tensor(0.0, device=DEVICE)
            loss_l1  = torch.tensor(0.0, device=DEVICE)

        loss_total = loss_mse + loss_dag + loss_l1

        # Physical losses (after warmup)
        loss_r10  = torch.tensor(0.0, device=DEVICE)
        loss_rx1d = torch.tensor(0.0, device=DEVICE)
        loss_cdd  = torch.tensor(0.0, device=DEVICE)
        loss_cc   = torch.tensor(0.0, device=DEVICE)
        if phys_active:
            # Full HR pred in log1p space
            full_pred = baseline_log + mu_total   # residual reconstruction (no delta yet at Stage 1)
            full_tgt  = baseline_log + target_residual
            loss_r10  = LAMBDA_R10MM  * loss_R10mm(full_pred, full_tgt, valid_mask=valid.float())
            loss_rx1d = LAMBDA_RX1DAY * loss_Rx1day(full_pred, full_tgt)
            loss_cdd  = LAMBDA_CDD    * loss_CDD(full_pred, full_tgt)
            loss_total = loss_total + loss_r10 + loss_rx1d + loss_cdd
            # CC loss : skip if T_850 not in batch (would need to thread it through)
            # For now, document as TODO -- requires lr_grid to carry T_850 explicitly

        loss_total.backward()
        torch.nn.utils.clip_grad_norm_(stage1_params, 1.0)
        optimizer_s1.step()

        epoch_losses['mse'].append(float(loss_mse.detach()))
        epoch_losses['dag'].append(float(loss_dag.detach()))
        epoch_losses['l1'].append(float(loss_l1.detach()))
        epoch_losses['r10mm'].append(float(loss_r10.detach()))
        epoch_losses['rx1day'].append(float(loss_rx1d.detach()))
        epoch_losses['cdd'].append(float(loss_cdd.detach()))
        epoch_losses['cc'].append(float(loss_cc.detach()))
        _n_batches_done += 1
        if SMOKE_MODE and _n_batches_done >= 30:
            break   # SMOKE : process only 30 train batches per epoch

    ep_time = time.time() - _t0
    avg = {k: float(np.mean(v)) if v else 0.0 for k, v in epoch_losses.items()}
    print(f'[ep{ep}/{STAGE1_EPOCHS}] mse={avg["mse"]:.4f} dag={avg["dag"]:.4f} l1={avg["l1"]:.4f} '
          f'r10={avg["r10mm"]:.4f} rx1d={avg["rx1day"]:.4f} cdd={avg["cdd"]:.4f} '
          f'(phys={phys_active}) time={ep_time:.0f}s n_batches={_n_batches_done}')

    stage1_history.append({
        'epoch': ep, 'epoch_time_s': ep_time, 'losses': avg,
        'lambda_l1': lam_l1, 'phys_active': phys_active,
        'n_batches': _n_batches_done,
    })

    # Save Stage 1 checkpoint each epoch
    payload_s1 = {
        'epoch': ep,
        'encoder_state_dict': encoder.state_dict(),
        'rcn_cell_state_dict': rcn_cell.state_dict(),
        'regression_head_state_dict': regression_head.state_dict(),
        'dual_path_state_dict': dual_path.state_dict(),
        'alpha_logit': alpha_logit.detach().cpu(),
        'optimizer_state_dict': optimizer_s1.state_dict(),
        'history': stage1_history,
        'pre_reg': PRE_REG_RECORD,
    }
    torch.save(payload_s1, CKPT_STAGE1_LAST)

    if SMOKE_MODE and ep >= STAGE1_EPOCHS:
        break

# Freeze Stage 1 after training
for m in [encoder, rcn_cell, regression_head, dual_path]:
    for p in m.parameters(): p.requires_grad_(False)
    m.eval()

# Print final A_dag stats (Q_phys interpretability)
_rcn_core = rcn_cell._orig_mod if hasattr(rcn_cell, '_orig_mod') else rcn_cell
if hasattr(_rcn_core, 'A_dag'):
    _A = _rcn_core.A_dag.detach()
    print(f'[Cell 8] Final A_dag : shape={tuple(_A.shape)} norm={_A.norm():.4f} '
          f'asymmetry={(_A - _A.T).abs().mean():.4f}')

print(f'[Cell 8] Stage 1 training complete. Checkpoint : {CKPT_STAGE1_LAST}')


In [ ]:
# === Cell 9 : Precompute mu_total cache (Stage 1 FROZEN forward) ===
# One-time pass on the train_dataset to cache (mu_total, baseline_log, delta_target).
# Used by Stage 2 training (cached -> 70x faster than recomputing Stage 1 each batch).
import time

if STAGE1_CACHE_PATH.exists() and not SMOKE_MODE:
    print(f'[Cell 9] Cache exists, loading : {STAGE1_CACHE_PATH}')
    cache = torch.load(STAGE1_CACHE_PATH, map_location='cpu', weights_only=False)
    cache = {k: v.contiguous().clone() for k, v in cache.items()}
else:
    print(f'[Cell 9] Precomputing mu_total cache from train_dataset (SMOKE={SMOKE_MODE})...')
    _t0 = time.time()
    mu_list, base_list, delta_list, valid_list = [], [], [], []
    _count = 0
    for sample in train_dataset:
        batch = convert_sample_to_batch(sample, builder, DEVICE)
        with torch.no_grad():
            mu_A, mu_total, mu_B, gate, baseline_log, target_residual = _stage1_forward(batch)
            delta_target = target_residual - mu_total
            valid = torch.isfinite(target_residual)
        mu_list.append(mu_total.squeeze(0).cpu())
        base_list.append(baseline_log.squeeze(0).cpu())
        delta_list.append(torch.nan_to_num(delta_target, nan=0.0).squeeze(0).cpu())
        valid_list.append(valid.squeeze(0).cpu())
        _count += 1
        if SMOKE_MODE and _count >= 30: break
        if _count % 500 == 0:
            print(f'  cached {_count} samples ({(time.time()-_t0)/60:.1f} min)')
    cache = {
        'mu_HR':        torch.stack(mu_list, dim=0),
        'baseline_log': torch.stack(base_list, dim=0),
        'delta_target': torch.stack(delta_list, dim=0),
        'valid_mask':   torch.stack(valid_list, dim=0),
    }
    torch.save(cache, STAGE1_CACHE_PATH)
    print(f'[Cell 9] Cache built in {time.time()-_t0:.1f}s : {STAGE1_CACHE_PATH}')

# Integrity check (Expert IA from previous audit)
for k in ('mu_HR', 'baseline_log', 'delta_target'):
    if not torch.isfinite(cache[k]).all():
        raise RuntimeError(f'cache[{k!r}] contains NaN/Inf -- delete and rerun')

N_CACHE = int(cache['mu_HR'].shape[0])
print(f'[Cell 9] Cache shapes :')
for k, v in cache.items():
    print(f'  {k} = {tuple(v.shape)}')
print(f'[Cell 9] Total cached samples : {N_CACHE}')

# Cached dataset for Stage 2
class _CachedDataset(torch.utils.data.Dataset):
    def __init__(self, cache, indices=None):
        self.mu_HR = cache['mu_HR']
        self.baseline_log = cache['baseline_log']
        self.delta_target = cache['delta_target']
        self.valid_mask = cache['valid_mask']
        self.indices = indices if indices is not None else list(range(len(self.mu_HR)))
    def __len__(self): return len(self.indices)
    def __getitem__(self, i):
        idx = self.indices[i]
        return {
            'mu_HR': self.mu_HR[idx],
            'baseline_log': self.baseline_log[idx],
            'delta_target': self.delta_target[idx],
            'valid_mask': self.valid_mask[idx],
        }

_N_TRAIN_S2 = int(N_CACHE * 0.9)
train_cached = _CachedDataset(cache, list(range(_N_TRAIN_S2)))
val_cached   = _CachedDataset(cache, list(range(_N_TRAIN_S2, N_CACHE)))
train_cached_dataloader = torch.utils.data.DataLoader(
    train_cached, batch_size=STAGE2_BATCH_SIZE, shuffle=True,
    num_workers=0, pin_memory=False, drop_last=False,
)
val_cached_dataloader = torch.utils.data.DataLoader(
    val_cached, batch_size=STAGE2_BATCH_SIZE, shuffle=False,
    num_workers=0, pin_memory=False, drop_last=False,
)
print(f'[Cell 9] Stage 2 dataloaders : train={len(train_cached)}  val={len(val_cached)}  bs={STAGE2_BATCH_SIZE}')


In [ ]:
# === Cell 10 : Stage 2 build (UNet 50M EDM + Dispersive hook + multi-EMA) ===
import copy
from st_cdgm.models import CausalDiffusionDecoder
from st_cdgm.models.edm_preconditioner import EDMConfig
from omegaconf import OmegaConf as _OC

# Probe HR channels
_probe = next(iter(val_dataset))
hr_channels = int(_probe['residual'].shape[1])

UNET_KW = _OC.to_container(CONFIG.diffusion.unet_kwargs, resolve=True)
for _k in ('down_block_types', 'up_block_types'):
    if _k in UNET_KW and isinstance(UNET_KW[_k], list):
        UNET_KW[_k] = tuple(UNET_KW[_k])
UNET_KW['projection_class_embeddings_input_dim'] = num_vars * int(CONFIG.diffusion.conditioning_dim)

edm_cfg = EDMConfig.from_yaml_dict(CONFIG.diffusion.get('edm', {}))

def _build_stage2_decoder():
    d = CausalDiffusionDecoder(
        in_channels=hr_channels,
        conditioning_dim=CONFIG.diffusion.conditioning_dim,
        height=int(CONFIG.diffusion.height), width=int(CONFIG.diffusion.width),
        unet_kwargs=UNET_KW,
        scheduler_type=str(CONFIG.diffusion.scheduler_type),
        use_gradient_checkpointing=True,
        conv_padding_mode=str(CONFIG.diffusion.get('conv_padding_mode', 'zeros')),
        anti_checkerboard=bool(CONFIG.diffusion.get('anti_checkerboard', False)),
        edm_config=edm_cfg, causal_concat=True,
    ).to(DEVICE)
    d.edm_config.sigma_data = float(SIGMA_DATA_NEW)
    return d

# Live decoder (the one being trained)
diff_decoder = _build_stage2_decoder()
n_params = sum(p.numel() for p in diff_decoder.parameters())
print(f'[Cell 10] Stage 2 decoder built : {n_params:,} params (sigma_data={SIGMA_DATA_NEW})')

# Multi-EMA models (Expert ML : 3 EMA decays + post-hoc sweep at eval)
ema_decoders = []
for decay in EMA_DECAYS:
    ema = _build_stage2_decoder()
    ema.load_state_dict(diff_decoder.state_dict())
    ema.eval()
    for p in ema.parameters(): p.requires_grad_(False)
    ema._ema_decay = decay
    ema._ema_step_counter = 0
    ema_decoders.append(ema)
print(f'[Cell 10] Multi-EMA decoders built : decays = {EMA_DECAYS}')

# ---- Dispersive Loss hook on UNet mid-block (Wang & He 2025) ----
# Captures mid-block features for the dispersive penalty term.
dispersive_features = {'mid': None}

def _dispersive_hook(module, inputs, output):
    # Module output is a Tensor (mid-block residual). Store a detached-noncausal handle.
    if isinstance(output, tuple):
        feat = output[0]
    else:
        feat = output
    dispersive_features['mid'] = feat

# Attach hook to UNet mid_block (diffusers UNet2DConditionModel exposes .mid_block)
try:
    _mid_block = diff_decoder.unet.mid_block
    _disp_handle = _mid_block.register_forward_hook(_dispersive_hook)
    print(f'[Cell 10] Dispersive Loss hook registered on mid_block (lambda={DISPERSIVE_LAMBDA}, tau={DISPERSIVE_TAU})')
except AttributeError:
    print(f'[Cell 10] WARNING : unet.mid_block not found, Dispersive Loss disabled')
    _disp_handle = None

def loss_dispersive(features, tau=DISPERSIVE_TAU):
    """Dispersive Loss (Wang & He 2025, arXiv 2506.09027).
    Encourages batch-wise feature diversity via contrastive repulsion.
    L = -log(E[K(z_i, z_j) / tau])  where K is a Gaussian kernel.
    """
    if features is None or features.dim() < 2:
        return torch.tensor(0.0, device=DEVICE)
    B = features.shape[0]
    if B < 2:
        return torch.tensor(0.0, device=DEVICE)
    flat = features.flatten(start_dim=1)
    # Pairwise distance (cosine-like via normalized dot product)
    flat = flat / (flat.norm(dim=1, keepdim=True) + 1e-8)
    sim = (flat @ flat.T) / tau   # [B, B]
    # Off-diagonal repulsion : we want low pairwise similarity
    mask = ~torch.eye(B, dtype=torch.bool, device=sim.device)
    off_diag = sim[mask]
    # -log(softmax-like form) : encourages off-diag to be small
    return torch.logsumexp(off_diag.view(B, B-1), dim=1).mean()

# ---- Min-SNR-gamma weighting (Hang ICCV 2023) ----
def _min_snr_weight(sigma, sigma_data, gamma):
    """w_min_snr(sigma) = min(SNR, gamma) / SNR  where SNR = sigma_d^2 / sigma^2.
    Final weight = w_min_snr * lambda_karras (combined per Math expert)."""
    snr = (sigma_data / sigma.clamp_min(1e-8)) ** 2
    return (torch.clamp(snr, max=gamma) / snr).view(-1, 1, 1, 1)

# ---- Tail weight (4, 12) on extreme pixels ----
def _tail_weight(hr_log_recon, tau95_mm=15.0, tau99_mm=35.0, w95=TAIL_WEIGHT_P95, w99=TAIL_WEIGHT_P99):
    import math
    tau95 = math.log1p(tau95_mm)
    tau99 = math.log1p(tau99_mm)
    return (1.0 + (w95 - 1.0) * (hr_log_recon > tau95).float()
                + (w99 - w95) * (hr_log_recon > tau99).float())

print(f'[Cell 10] Min-SNR weighting ready (gamma={MIN_SNR_GAMMA})')
print(f'[Cell 10] Tail weight ready ({TAIL_WEIGHT_P95}, {TAIL_WEIGHT_P99}) on pixels >15/>35 mm/day')


In [ ]:
# === Cell 11 : Stage 2 training (Min-SNR + tail + Dispersive + cond_drop + multi-EMA) ===
# Inline implementation -- adapts train_epoch_stage2_cached pattern with all the fixes.
from st_cdgm.models.edm_preconditioner import sample_training_sigma
import time

# Optimizer (Stage 2 live + alpha_logit)
stage2_params = list(diff_decoder.parameters()) + [alpha_logit]
optimizer_s2 = torch.optim.AdamW(
    stage2_params,
    lr=STAGE2_LR, betas=(0.9, 0.999), weight_decay=STAGE2_WEIGHT_DECAY,
)

stage2_history = []
start_epoch_s2 = 1

# Resume support
if CKPT_STAGE2_LAST.exists() and not SMOKE_MODE:
    print(f'[Cell 11] RESUME from {CKPT_STAGE2_LAST}')
    _ck = torch.load(CKPT_STAGE2_LAST, map_location=DEVICE, weights_only=False)
    diff_decoder.load_state_dict(_ck['diffusion_state_dict'])
    if 'ema_state_dicts' in _ck:
        for i, ema in enumerate(ema_decoders):
            if i < len(_ck['ema_state_dicts']):
                ema.load_state_dict(_ck['ema_state_dicts'][i])
    if _ck.get('alpha_logit') is not None:
        alpha_logit.data = _ck['alpha_logit'].to(DEVICE)
    if _ck.get('optimizer_state_dict') is not None:
        try: optimizer_s2.load_state_dict(_ck['optimizer_state_dict'])
        except Exception as e: print(f'  optimizer resume failed : {e}')
    start_epoch_s2 = int(_ck.get('epoch', 0)) + 1
    stage2_history = list(_ck.get('history', []))
    print(f'  Resumed at epoch {start_epoch_s2}/{STAGE2_EPOCHS}')

print(f'[Cell 11] Stage 2 training : epochs {start_epoch_s2}..{STAGE2_EPOCHS} (SMOKE={SMOKE_MODE})')

def _ema_update_all(decoders, decays, live_model):
    """Multi-EMA update : in-place mul + add on params, copy on buffers."""
    with torch.no_grad():
        for ema, decay in zip(decoders, decays):
            for p_e, p_l in zip(ema.parameters(), live_model.parameters()):
                p_e.data.mul_(decay).add_(p_l.data, alpha=1.0 - decay)
            for b_e, b_l in zip(ema.buffers(), live_model.buffers()):
                b_e.data.copy_(b_l.data)
            ema._ema_step_counter += 1

for ep in range(start_epoch_s2, STAGE2_EPOCHS + 1):
    _t0 = time.time()
    diff_decoder.train()

    losses_log = {'edm': [], 'disp': [], 'alpha_reg': [], 'total': []}
    n_batches_seen = 0

    for batch in train_cached_dataloader:
        mu_HR = batch['mu_HR'].to(DEVICE, non_blocking=True)
        baseline_log = batch['baseline_log'].to(DEVICE, non_blocking=True)
        delta_target = batch['delta_target'].to(DEVICE, non_blocking=True)

        # Conditioning dropout (CFG compatibility, CorrDiff Nature CEE 2025)
        # When mu_HR is dropped, target also adjusted (Ho & Salimans branch correctness)
        mu_HR_used = mu_HR
        delta_target_used = delta_target
        if COND_DROPOUT_P > 0:
            B = mu_HR.shape[0]
            drop_mask = (torch.rand(B, device=DEVICE) < COND_DROPOUT_P).view(B, 1, 1, 1)
            if drop_mask.any():
                mu_HR_used = torch.where(drop_mask, torch.zeros_like(mu_HR), mu_HR)
                delta_target_used = torch.where(drop_mask, delta_target + mu_HR, delta_target)

        optimizer_s2.zero_grad(set_to_none=True)
        with torch.autocast('cuda', dtype=torch.bfloat16):
            B = delta_target_used.shape[0]
            sigma = sample_training_sigma(B, P_mean=edm_cfg.P_mean, P_std=edm_cfg.P_std,
                                            device=DEVICE, dtype=delta_target_used.dtype)
            noise = torch.randn_like(delta_target_used)
            y_noisy = delta_target_used + sigma.view(-1, 1, 1, 1) * noise

            # Forward EDM (this triggers the dispersive hook on mid_block)
            D_y = diff_decoder.forward_edm(
                y_noisy, sigma, conditioning=None, conditioning_spatial=None,
                mu_HR=mu_HR_used, baseline_log=baseline_log,
            )

            # ----- Combined loss weight : Karras lambda * Min-SNR gamma -----
            lambda_karras = (sigma**2 + edm_cfg.sigma_data**2) / (sigma * edm_cfg.sigma_data)**2
            min_snr_w = _min_snr_weight(sigma, edm_cfg.sigma_data, MIN_SNR_GAMMA)
            w_total = lambda_karras.view(-1, 1, 1, 1) * min_snr_w

            # ----- Tail weight on extreme pixels -----
            hr_log_recon = delta_target_used + mu_HR_used + baseline_log   # full HR in log1p
            tail_w = _tail_weight(hr_log_recon).detach()
            # Clip combined weight to avoid extreme values (Math : W_max=50)
            w_combined = (w_total * tail_w).clamp(max=50.0)

            sq_err = (D_y - delta_target_used) ** 2
            loss_edm = (w_combined * sq_err).mean()

            # ----- Dispersive Loss on mid-block features -----
            loss_disp = loss_dispersive(dispersive_features.get('mid', None))

            # ----- Alpha regularization -----
            alpha = torch.sigmoid(alpha_logit)
            loss_alpha = loss_alpha_reg(alpha)

            loss_total = loss_edm + DISPERSIVE_LAMBDA * loss_disp + loss_alpha

        loss_total.backward()
        torch.nn.utils.clip_grad_norm_(stage2_params, STAGE2_GRADIENT_CLIP)
        optimizer_s2.step()

        # Multi-EMA update
        _ema_update_all(ema_decoders, EMA_DECAYS, diff_decoder)

        losses_log['edm'].append(float(loss_edm.detach()))
        losses_log['disp'].append(float(loss_disp.detach()))
        losses_log['alpha_reg'].append(float(loss_alpha.detach()))
        losses_log['total'].append(float(loss_total.detach()))
        n_batches_seen += 1

        if SMOKE_MODE and n_batches_seen >= 10:
            break

    ep_time = time.time() - _t0
    avg = {k: float(np.mean(v)) if v else 0.0 for k, v in losses_log.items()}
    cur_alpha = float(torch.sigmoid(alpha_logit).detach())
    print(f'[ep{ep}/{STAGE2_EPOCHS}] edm={avg["edm"]:.4f} disp={avg["disp"]:.4f} '
          f'alpha_reg={avg["alpha_reg"]:.4f} total={avg["total"]:.4f} '
          f'alpha={cur_alpha:.4f} time={ep_time:.0f}s n_batches={n_batches_seen}')

    stage2_history.append({
        'epoch': ep, 'epoch_time_s': ep_time, 'losses': avg,
        'alpha': cur_alpha, 'n_batches': n_batches_seen,
    })

    # Save Stage 2 checkpoint each epoch
    payload_s2 = {
        'epoch': ep,
        'diffusion_state_dict': diff_decoder.state_dict(),
        'ema_state_dicts': [ema.state_dict() for ema in ema_decoders],
        'ema_decays': EMA_DECAYS,
        'ema_step_counters': [ema._ema_step_counter for ema in ema_decoders],
        'alpha_logit': alpha_logit.detach().cpu(),
        'optimizer_state_dict': optimizer_s2.state_dict(),
        'history': stage2_history,
        'sigma_data': float(SIGMA_DATA_NEW),
        'pre_reg': PRE_REG_RECORD,
    }
    torch.save(payload_s2, CKPT_STAGE2_LAST)

    if SMOKE_MODE and ep >= STAGE2_EPOCHS:
        break

# Detach hook
if _disp_handle is not None:
    _disp_handle.remove()

# Set all to eval at the end
diff_decoder.eval()
for ema in ema_decoders:
    ema.eval()

print(f'[Cell 11] Stage 2 training complete')
print(f'[Cell 11] Final alpha = {float(torch.sigmoid(alpha_logit)):.4f}')
print(f'[Cell 11] Final EMA step counters : {[ema._ema_step_counter for ema in ema_decoders]}')
print(f'[Cell 11] Checkpoint : {CKPT_STAGE2_LAST}')


## Cells 12-15 : Eval BS30 unifié + 3-way comparison (à implémenter dans commit suivant)

- Cell 12 : Sampling N=64 × K=128 × 32 steps avec Limited-Interval Guidance + post-hoc EMA sweep
- Cell 13 : Métriques (3 conventions F1, CSI, SEDI, FSS, CRPS, Pearson, RMSE, RAPSD) + indices climatiques sur 730 jours entiers
- Cell 14 : Paired permutation test + bootstrap BCa + Holm-Bonferroni
- Cell 15 : Tableau 3-way (Phase 8 vs V5 vs noncausal) + JSON publication-ready + plots